In [2]:
import pandas as pd
import numpy as np

# =========================
# 1. Load the dataset
# =========================
file_path = "/content/fts_requirements_funding_global.csv"  # change if needed
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# =========================
# 2. Keep only useful columns
# =========================
cols_to_keep = [
    "countryCode",
    "name",
    "year",
    "requirements",
    "funding",
    "percentFunded",
    "startDate",
    "endDate"
]

df = df[cols_to_keep].copy()

# =========================
# 3. Rename columns
# =========================
df = df.rename(columns={
    "countryCode": "CountryCode",
    "name": "Country",
    "year": "Year",
    "requirements": "RequiredFunding",
    "funding": "ReceivedFunding",
    "percentFunded": "PercentFunded",
    "startDate": "StartDate",
    "endDate": "EndDate"
})

# =========================
# 4. Convert data types
# =========================
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["RequiredFunding"] = pd.to_numeric(df["RequiredFunding"], errors="coerce")
df["ReceivedFunding"] = pd.to_numeric(df["ReceivedFunding"], errors="coerce")
df["PercentFunded"] = pd.to_numeric(df["PercentFunded"], errors="coerce")

df["StartDate"] = pd.to_datetime(df["StartDate"], errors="coerce")
df["EndDate"] = pd.to_datetime(df["EndDate"], errors="coerce")

# =========================
# 5. Basic cleaning
# =========================
# Remove rows missing key fields
df = df.dropna(subset=["Country", "Year", "RequiredFunding", "ReceivedFunding"])

# Keep only 2015+
df = df[df["Year"] >= 2015]

# Remove rows with impossible negative values
df = df[(df["RequiredFunding"] >= 0) & (df["ReceivedFunding"] >= 0)]

# Clean country names
df["Country"] = df["Country"].astype(str).str.strip()

# =========================
# 6. Fill or recalculate PercentFunded
# =========================
# If PercentFunded is missing, calculate it
df["PercentFunded"] = np.where(
    df["PercentFunded"].isna() & (df["RequiredFunding"] > 0),
    (df["ReceivedFunding"] / df["RequiredFunding"]) * 100,
    df["PercentFunded"]
)

# =========================
# 7. Create Funding Gap
# =========================
df["FundingGap"] = df["RequiredFunding"] - df["ReceivedFunding"]

# Optional: if funding exceeds requirements, keep negative gap as 0
df["FundingGap"] = df["FundingGap"].clip(lower=0)

# =========================
# 8. Sort the data
# =========================
df = df.sort_values(["Year", "Country"]).reset_index(drop=True)

# =========================
# 9. Save cleaned file
# =========================
output_file = "/content/fts_requirements_funding_cleaned.csv"
df.to_csv(output_file, index=False)

print("\nCleaned shape:", df.shape)
print("\nPreview:")
print(df.head())

print(f"\nCleaned file saved as: {output_file}")

Original shape: (3779, 12)

Columns:
['countryCode', 'id', 'name', 'code', 'typeId', 'typeName', 'startDate', 'endDate', 'year', 'requirements', 'funding', 'percentFunded']

Cleaned shape: (714, 9)

Preview:
  CountryCode                                        Country  Year  \
0         AFG       Afghanistan Strategic Response Plan 2015  2015   
1         BFA  Burkina Faso Plan de Réponse Stratégique 2015  2015   
2         CMR      Cameroun Plan de Réponse Stratégique 2015  2015   
3         PRK            DPR Korea Needs and Priorities 2015  2015   
4         DJI      Djibouti Plan de Réponse Stratégique 2015  2015   

   RequiredFunding  ReceivedFunding  PercentFunded  StartDate    EndDate  \
0      416666132.0      319427000.0           77.0 2015-01-01 2015-12-31   
1       98761764.0       30631658.0           31.0 2015-01-01 2015-12-31   
2      264023457.0      129246961.0           49.0 2015-01-01 2015-12-31   
3      110895000.0       23415905.0           21.0 2015-01-01 2015-

In [3]:
focus_countries = [
    "Afghanistan",
    "Bangladesh",
    "Ethiopia",
    "Somalia",
    "South Sudan",
    "Sudan",
    "Syrian Arab Republic",
    "Ukraine",
    "Yemen",
    "Democratic Republic of the Congo"
]

df_focus = df[df["Country"].isin(focus_countries)].copy()

output_focus_file = "/content/fts_requirements_funding_focus_countries.csv"
df_focus.to_csv(output_focus_file, index=False)

print("Focus dataset shape:", df_focus.shape)
print(f"Focus file saved as: {output_focus_file}")

Focus dataset shape: (0, 9)
Focus file saved as: /content/fts_requirements_funding_focus_countries.csv


In [4]:
summary = (
    df.groupby(["Country", "CountryCode", "Year"], as_index=False)
      .agg({
          "RequiredFunding": "sum",
          "ReceivedFunding": "sum",
          "FundingGap": "sum"
      })
)

summary["PercentFunded"] = np.where(
    summary["RequiredFunding"] > 0,
    (summary["ReceivedFunding"] / summary["RequiredFunding"]) * 100,
    np.nan
)

summary_file = "/content/fts_requirements_funding_summary_country_year.csv"
summary.to_csv(summary_file, index=False)

print("Summary dataset shape:", summary.shape)
print(summary.head())
print(f"Summary file saved as: {summary_file}")

Summary dataset shape: (714, 7)
                                             Country CountryCode  Year  \
0                      Afghanistan Flash Appeal 2016         AFG  2016   
1                      Afghanistan Flash Appeal 2021         AFG  2021   
2  Afghanistan Humanitarian Needs and Response Pl...         AFG  2024   
3  Afghanistan Humanitarian Needs and Response Pl...         AFG  2025   
4  Afghanistan Humanitarian Needs and Response Pl...         AFG  2026   

   RequiredFunding  ReceivedFunding    FundingGap  PercentFunded  
0     1.521000e+08     6.849499e+07  8.360501e+07      45.032865  
1     6.062612e+08     9.982428e+08  0.000000e+00     164.655569  
2     3.059588e+09     1.584154e+09  1.475434e+09      51.776700  
3     2.416812e+09     1.006622e+09  1.410190e+09      41.650815  
4     1.714181e+09     1.842984e+08  1.529883e+09      10.751395  
Summary file saved as: /content/fts_requirements_funding_summary_country_year.csv
